# Comparing All Three: NB+BoW vs NB+TF-IDF vs SVM+TF-IDF

Step 4, the actual point of this project. Loads the cross-validated metrics saved by
`02_naive_bayes.ipynb` and `03_svm.ipynb` (same `StratifiedKFold(random_state=42)` folds in all
three, so this is comparing on identical splits, not just similar ones) and puts them side by
side.

## Imports

In [1]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Load Saved Metrics

In [2]:
OUTPUTS_DIR = os.path.join("..", "data", "outputs")

METHOD_FILES = {
    "NB + Bag-of-Words": "nb_bow_metrics.json",
    "NB + TF-IDF": "nb_tfidf_metrics.json",
    "SVM + TF-IDF": "svm_tfidf_metrics.json",
}

results = {}
for label, filename in METHOD_FILES.items():
    with open(os.path.join(OUTPUTS_DIR, filename), encoding="utf-8") as f:
        results[label] = json.load(f)

{label: r["model"] + " / " + r["vectorizer"] for label, r in results.items()}

## Comparison Table

In [3]:
METRICS = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]

table = pd.DataFrame(
    {label: [f"{r['mean'][m]:.4f} ± {r['std'][m]:.4f}" for m in METRICS] for label, r in results.items()},
    index=METRICS,
)
table

## Chart

A grouped bar chart is the right form here: comparing a handful of named methods (identity, fixed
categorical order — not a rank) across a small, fixed set of magnitude metrics, all on the same
0-1 scale, so one shared y-axis is correct (no dual axis). Error bars show the fold-to-fold std
already computed in Steps 2-3, not just the mean — a method that wins on average but wobbles a lot
between folds is a different finding than one that wins consistently.

Colors are assigned to methods in a fixed order and reused consistently (never re-cycled if a
method were added or removed) — blue / orange / green, the same trio scikit-learn's own
`Naive Bayes`, `LDA`, `QDA` matplotlib docs example uses, chosen for being distinguishable under
the common forms of color blindness (blue vs. orange is preserved under both deuteranopia and
protanopia) rather than for taste.

In [4]:
METHOD_COLORS = {
    "NB + Bag-of-Words": "#4C72B0",
    "NB + TF-IDF": "#DD8452",
    "SVM + TF-IDF": "#55A868",
}
METRIC_LABELS = ["Accuracy", "Precision\n(macro)", "Recall\n(macro)", "F1\n(macro)"]

x = np.arange(len(METRICS))
n_methods = len(results)
bar_width = 0.8 / n_methods

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.set_axisbelow(True)
ax.yaxis.grid(True, color="#DDDDDD", linewidth=0.8)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)

for i, (label, r) in enumerate(results.items()):
    means = [r["mean"][m] for m in METRICS]
    stds = [r["std"][m] for m in METRICS]
    offset = (i - (n_methods - 1) / 2) * bar_width
    ax.bar(
        x + offset, means, width=bar_width * 0.9,
        yerr=stds, capsize=3,
        color=METHOD_COLORS[label], label=label,
    )

ax.set_xticks(x)
ax.set_xticklabels(METRIC_LABELS)
ax.set_ylim(0, 1.0)
ax.set_ylabel("Score (5-fold CV mean ± std)")
ax.set_title("Naive Bayes vs SVM: Bag-of-Words vs TF-IDF")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()

fig_path = os.path.join(OUTPUTS_DIR, "comparison.png")
fig.savefig(fig_path, dpi=150)
print(f"Saved {fig_path}")
plt.show()

## Conclusion

Checked against the real cross-validated numbers, not assumed from theory alone:

1. **"NB wants BoW, not TF-IDF" held up** (Step 2): Bag-of-Words beat TF-IDF for `MultinomialNB`
   on every metric (macro-F1 0.777 vs 0.733), a gap bigger than either method's fold-to-fold std —
   a real effect on this dataset, not noise.
2. **SVM + TF-IDF beat both NB variants** (macro-F1 0.801 vs NB+BoW's 0.777 and NB+TF-IDF's 0.733)
   — consistent with SVM's larger hypothesis space (a linear decision boundary that can weigh every
   feature independently, rather than NB's conditional-independence assumption across all those
   bigram features).
3. **SVM was also the most *consistent* method**, not just the best on average: its fold-to-fold
   std (≈0.006 on macro-F1) was roughly a third of NB+BoW's (≈0.015) — worth knowing separately
   from the mean, since a method that wins on average but swings widely between folds is a weaker
   claim than one that wins and stays put.

**Overall: SVM + TF-IDF (bigrams) is the best of the three methods tested here**, both on average
performance and on stability across folds. NB + Bag-of-Words is a reasonable, cheaper runner-up if
training/inference cost mattered more than squeezing out the last few points of accuracy; NB +
TF-IDF is not competitive with either and mainly useful here as the empirical confirmation of why
NB isn't usually paired with TF-IDF in the first place.